#  Winning Jeopardy

Jeopardy is a popular TV show in the US where participants answer questions to win money. It's been running for many years, and is a major force in popular culture.

In this project, I will work with a dataset of Jeopardy questions to figure out some patterns in the questions that could help us win.

The dataset is named jeopardy.csv, and contains 20000 rows from the beginning of a full dataset of Jeopardy questions, which you can download [here](https://www.reddit.com/r/datasets/comments/1uyd0t/200000_jeopardy_questions_in_a_json_file). Here's the beginning of the file:

## Explore the dataset

In [1]:
# import neccessary module
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the csv file
jeopardy = pd.read_csv('../../dataset/jeopardy.csv')

In [2]:
# Print the first few rows
jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams


In [3]:
jeopardy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19999 entries, 0 to 19998
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Show Number  19999 non-null  int64 
 1    Air Date    19999 non-null  object
 2    Round       19999 non-null  object
 3    Category    19999 non-null  object
 4    Value       19663 non-null  object
 5    Question    19999 non-null  object
 6    Answer      19999 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.1+ MB


In [4]:
# Remove the spaces in front of column names
jeopardy.columns = jeopardy.columns.str.strip() 

In [5]:
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='object')

## Normalizing Text

Write a function to normalize questions and answers.

In [6]:
# normalization function
import re
def normalization(x):
    return re.sub('[^\w\s]', '',x).lower()

jeopardy['clean_question'] = jeopardy['Question'].apply(normalization)
jeopardy['clean_answer'] = jeopardy['Answer'].apply(normalization)
jeopardy['clean_question'].head()

0    for the last 8 years of his life galileo was u...
1    no 2 1912 olympian football star at carlisle i...
2    the city of yuma in this state has a record av...
3    in 1963 live on the art linkletter show this c...
4    signer of the dec of indep framer of the const...
Name: clean_question, dtype: object

## Normalizing Columns

In [7]:
# function to normalize dollar values

def normalize_dollar(x):
    try:
        x = int(str(x).replace('$', ''))
    except Exception:
        x = 0
    return x

jeopardy['clean_value'] = jeopardy['Value'].apply(normalize_dollar)
jeopardy['Air Date'] = pd.to_datetime(jeopardy['Air Date'])

In [8]:
jeopardy.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,clean_question,clean_answer,clean_value
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus,for the last 8 years of his life galileo was u...,copernicus,200
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe,no 2 1912 olympian football star at carlisle i...,jim thorpe,200
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona,the city of yuma in this state has a record av...,arizona,200
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's,in 1963 live on the art linkletter show this c...,mcdonalds,200
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams,signer of the dec of indep framer of the const...,john adams,200


In [9]:
jeopardy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19999 entries, 0 to 19998
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Show Number     19999 non-null  int64         
 1   Air Date        19999 non-null  datetime64[ns]
 2   Round           19999 non-null  object        
 3   Category        19999 non-null  object        
 4   Value           19663 non-null  object        
 5   Question        19999 non-null  object        
 6   Answer          19999 non-null  object        
 7   clean_question  19999 non-null  object        
 8   clean_answer    19999 non-null  object        
 9   clean_value     19999 non-null  int64         
dtypes: datetime64[ns](1), int64(2), object(7)
memory usage: 1.5+ MB


## Answers in Questions

In order to figure out whether to study past questions, study general knowledge, or not study it all, it would be helpful to figure out two things:

- How often the answer can be used for a question.
- How often questions are repeated.


In [10]:
# fucntion that takes a row in jeopardy

def series_split(series):
    split_answer = series['clean_answer'].split()
    split_question = series['clean_question'].split()
    
    match_count = 0
    if 'the' in split_answer:
        split_answer.remove('the')
    if len(split_answer) == 0:
        return 0
    for value in split_answer:
        if value in split_question:
            match_count += 1
    return match_count / len(split_answer)

jeopardy['answer_in_question'] = jeopardy.apply(series_split, axis = 1)
jeopardy['answer_in_question'].mean()

np.float64(0.05900196524977763)

## Recycled Questions
On average, the answer only makes up for about 6% of the question. This isn't a huge number, and it means that we probably can't just hope that hearing a question will enable us to determine the answer. We'll probably have to study.

In [11]:
question_overlap = list()
terms_used = set()
jeopardy = jeopardy.sort_values(by = 'Air Date')

for index, series in jeopardy.iterrows():
    split_question = series['clean_question'].split()
    split_question = [word for word in split_question if len(word) >= 6]
    match_count = 0
    for word in split_question:
        if word in terms_used:
            match_count += 1
    for word in split_question:
        terms_used.add(word)
    if len(split_question) > 0:
        match_count /=  len(split_question)
    question_overlap.append(match_count)
jeopardy['question_overlap'] = question_overlap
print(jeopardy['question_overlap'].mean())
            

0.6876235590919739


## Low Value vs. High Value Questions
There is about a `70%` overlap between terms in new questions and terms in old questions.  This only looks at a small set of questions, and it doesn't look at phrases — it looks at single terms.  This makes it relatively insignificant, but it does mean that it's worth looking more into the recycling of questions.

In [12]:
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer', 'clean_question', 'clean_answer', 'clean_value',
       'answer_in_question', 'question_overlap'],
      dtype='object')

In [17]:
# function that takes in a row from a Dataframe

def function_row(row):
    if row['clean_value'] > 800:
        value = 1
    else:
        value = 0
    return value
        
jeopardy['high_value'] = jeopardy.apply(function_row, axis = 1)

# function that take in a word
def word_count(word):
    low_count = 0
    high_count = 0
    for index, row in jeopardy.iterrows():
        if word in row['clean_question'].split(" "):
            if row['high_value'] == 1:
                high_count += 1
            else:
                low_count += 1
    return high_count, low_count


In [18]:
from random import choice
comparison_terms = [choice(list(terms_used)) for _ in range(10)]
observed_expected = list()
for term in comparison_terms:
    observed_expected.append(word_count(term))
        

In [19]:
observed_expected

[(1, 0),
 (0, 2),
 (0, 1),
 (1, 3),
 (0, 1),
 (1, 0),
 (3, 10),
 (0, 1),
 (1, 0),
 (0, 1)]

In [20]:
high_value_count = jeopardy[jeopardy['high_value'] == 1].shape[0]
low_value_count = jeopardy[jeopardy['high_value'] == 0].shape[0]
from scipy.stats import chisquare
chi_squared = list()
for item in observed_expected:
    total = sum(item)
    total_prop = total / jeopardy.shape[0]
    high_value_expected = total_prop * high_value_count
    low_value_expected = total_prop * low_value_count
    
    expected = np.array([high_value_expected, low_value_expected])
    observed = np.array([item[0], item[1]])
    chi_squared.append(chisquare(observed, expected))

chi_squared
    

[Power_divergenceResult(statistic=np.float64(3.022325020112631), pvalue=np.float64(0.08212564786568953)),
 Power_divergenceResult(statistic=np.float64(0.661742197378053), pvalue=np.float64(0.4159455550913673)),
 Power_divergenceResult(statistic=np.float64(0.3308710986890265), pvalue=np.float64(0.565146603267378)),
 Power_divergenceResult(statistic=np.float64(4.122707846712507e-05), pvalue=np.float64(0.9948769527982859)),
 Power_divergenceResult(statistic=np.float64(0.3308710986890265), pvalue=np.float64(0.565146603267378)),
 Power_divergenceResult(statistic=np.float64(3.022325020112631), pvalue=np.float64(0.08212564786568953)),
 Power_divergenceResult(statistic=np.float64(0.022156542301255255), pvalue=np.float64(0.8816714131683532)),
 Power_divergenceResult(statistic=np.float64(0.3308710986890265), pvalue=np.float64(0.565146603267378)),
 Power_divergenceResult(statistic=np.float64(3.022325020112631), pvalue=np.float64(0.08212564786568953)),
 Power_divergenceResult(statistic=np.float64(

## Chi-Squared Results

None of the terms had a significant difference in usage between high value and low value rows. Additionally, the frequencies were all lower than `5`, so the chi-squared test isn't as valid. It would be better to run this test with only terms that have higher frequencies.